# Data Cleaning
This notebook is used to apply the cleaning steps to the raw dataset and explain the reasons behind each transformation/decision.

In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.insert(0, '..')
from src.helpers import expand_dispatch_date, age_range_to_midpoint

df = pd.read_csv("../data/naloxone.csv")

## Renaming Columns to snake_case
The original column names use title case with spaces, which makes them a bit harder to work with in Python. When it comes to coding I prefer to use snake_case naming convention. Renaming everything to snake_case is a small change that keeps the rest of the code cleaner. In this step I also fixed a typo in `Naxolone Administrations`. The correct spelling is *Na**lo**xone*.

In [2]:
df = df.rename(columns=lambda x: x.replace(" ", "_").lower())

In [3]:
df = df.rename(columns={
    "naxolone_administrations": "naloxone administrations"
})

## Dropping Unecessary Columns
Three columns were dropped because they don't add much analytical value to the analysis:
- `id`: just a composite key made from `incident_number` and `patient_number`.

- `neighbourhood`: has 223 unique values, many with only one incident, which makes it hard to analyze.
    - `ward` already does a great job covering this grouped geographical information into 15 categories.

- `neighbourhood_id`: just a code for each neighbourhood we already dropped. 

In [4]:
df = df.drop(columns=[
    "id",
    "neighbourhood",
    "neighbourhood_id"
])

## Expanding `dispatch_date`
`dispatch_date` was stored as a plain string (`"2021-08-08T04:33:22"`), which means pandas treats it as text. The best way to filter / extract information from this column is converting it to real datetime data type. After parsing it to datetime, I extracted `year`, `month`, `day_of_week`, and `hour` as separate columns. These are temporal dimensions columns that I plan to use to groupby/filter in the analysis. I also kept a `date` column (date only, no time) as a reference. The original `dispatch_date` was dropped after extracting the other columns out since the needed information lives in those new columns.

In [5]:
df = expand_dispatch_date(df)

## Standardizing Missing Values
The `age` and `gender` columns uses both real NaN (missing values) and also the string `"Unknown"` as ways of showing that data is missing. This way, pandas is treating only the real NaN as missing values, so I replaced `"Unknown"` entries by real missing values.

In [6]:
df[["age", "gender"]] = df[["age", "gender"]].replace("Unknown", np.nan)

## Creating `age_midpoint`
`age` stores ranges like `"25 to 29"` instead of a single number, which is fine for categorical plots, but unusable when you need a numeric value to do calculations. I then created a new column, `age_midpoint`, to hold the mean (midpoint) between the lower and upper boundary of the range. Having both columns in the dataset is useful, so it covers both cases.

In [7]:
df["age_midpoint"] = df["age"].apply(age_range_to_midpoint)

## Transformations Summary
- Rename columns: converted all column names to snake_case; fixed typo `"naxolone"` -> `"naloxone"`.

- Drop columns: removed `id`, `neighbourhood`, and `neighobourhood_id`.

- Expand `dispatch_date`: parsed `dispatch_date` to datetime; extracted `year`, `month`, `day_of_week`, `hour`, and `date` as separate columns; dropped original column `dispatch_date`.

- Standardize missing values: replaced `"Unknown"` strings in `age` and `gender` columns with real `NaN`.

- Create `age_midpoint` column: calculated numeric midpoint from age range strings to float (ex: `"25 to 29"` -> `27.0`); kept original column `age` with the strings age ranges. 

## Clean Dataset

In [9]:
df.head(10)

,incident_number,patient_number,age,gender,ward,naloxone administrations,date,year,month,day_of_week,hour,age_midpoint
0,2021095567,1,30 to 34,Male,NaN,1,2021-08-08,2021,8,6,4,32.0
1,2021095606,1,55 to 59,Female,Fort Rouge - East Fort Garry,1,2021-08-08,2021,8,6,7,57.0
2,2021095828,1,50 to 54,Female,Fort Rouge - East Fort Garry,1,2021-08-08,2021,8,6,16,52.0
3,2021095933,1,35 to 39,Female,NaN,2,2021-08-08,2021,8,6,20,37.0
4,2021095940,1,25 to 29,Male,Point Douglas,1,2021-08-08,2021,8,6,21,27.0
5,2021095986,1,25 to 29,Male,Daniel McIntyre,1,2021-08-08,2021,8,6,23,27.0
6,2021095998,1,20 to 24,Male,St. James,1,2021-08-08,2021,8,6,23,22.0
7,2021096027,1,20 to 24,Male,Mynarski,2,2021-08-09,2021,8,0,1,22.0
8,2021096313,1,60 to 64,Female,Daniel McIntyre,1,2021-08-09,2021,8,0,15,62.0
9,2021096328,1,25 to 29,Male,Mynarski,1,2021-08-09,2021,8,0,16,27.0
